# Paper measures — computed from logs alone

The dry-run contract from `docs/deployment_logging_plan.md`: **every number in the
paper's Measures section must be computable from the logs of one meal.** Each section
below computes one family of measures and registers a scorecard entry:

- **GREEN** — computed from this meal's logs.
- **PARTIAL** — computed, but with a caveat noted (proxy method, or needs multiple days).
- **RED** — the required log stream is absent from *this* meal. A RED here is the
  notebook doing its job: it names the gap. Meals recorded at ≥ `48d3308`
  (2026-07-16) emit every stream this notebook reads.
- **SKIP** — environment limitation (e.g. no ROS for bag reading), not a logging gap.

Point the paths cell at any meal and re-run. Keep this notebook in the repo — it is
the start of the paper's analysis pipeline.

In [1]:
# ---- inputs: point these at one meal ---------------------------------------
from pathlib import Path

REPO = Path("~/deployment_ws/src/feeding-deployment").expanduser()
LOG = REPO / "src/feeding_deployment/integration/log"

USER_DIR = LOG / "jul15_test4"                       # log/<user>/
DAY_DIR = USER_DIR / "day_01"                        # the meal's day dir
SESSION_BUNDLE = LOG / "system_logs/session_20260715_220058_jul15-test3"
MEAL_BAG_DIR = Path("/data/feeding_dataset/bags/meal_20260715_222320")

for p in (USER_DIR, DAY_DIR, SESSION_BUNDLE):
    assert p.exists(), f"missing: {p}"
print("inputs ok")

inputs ok


In [2]:
# ---- loaders + scorecard ----------------------------------------------------
import json, csv, math, statistics, collections

def jsonl(path):
    if not Path(path).exists():
        return []
    out = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return out

def read_csv(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

events = jsonl(DAY_DIR / "events.jsonl")
user_inputs = jsonl(DAY_DIR / "user_inputs.jsonl")
images_index = jsonl(DAY_DIR / "images_index.jsonl")
metadata = json.loads((DAY_DIR / "metadata.json").read_text()) if (DAY_DIR / "metadata.json").exists() else {}

by_cat = collections.defaultdict(list)
for e in events:
    by_cat[e.get("category", "?")].append(e)

navlogs = sorted((SESSION_BUNDLE / "compute").glob("navlog_*"))
NAVLOG = navlogs[-1] if navlogs else None

SCORECARD = {}
def score(measure, status, note=""):
    SCORECARD[measure] = (status, note)
    print(f"[{status:7s}] {measure}" + (f" — {note}" if note else ""))

print(f"{len(events)} events in {sorted(by_cat)}")
print(f"{len(user_inputs)} user inputs; navlog: {NAVLOG.name if NAVLOG else 'MISSING'}")

13 events in ['preference_asked', 'preference_color_recorded', 'preference_finalized', 'preference_nav_offset_recorded', 'preference_predicted', 'task_command']
50 user inputs; navlog: navlog_20260715_220103


## 1. Preference learning efficiency (real-world analog of Table II)

First-prediction accuracy per dim (initial `preference_predicted` bundle vs the
ground truth the ask flow settled on), corrections per dim, and correction
propagation (`preference_repredicted`: which open dims moved after each trigger).

In [3]:
pred_events = by_cat.get("preference_predicted", [])
asked = by_cat.get("preference_asked", [])
repredicted = by_cat.get("preference_repredicted", [])

if pred_events and asked:
    initial = pred_events[0].get("predicted_bundle", {})
    rows, n_match, n_dims = [], 0, 0
    for a in asked:
        for d in a.get("dims", []):
            truth = a.get("ground_truth", {}).get(d)
            predicted = initial.get(d)
            corrected = d in a.get("corrected", [])
            match = predicted == truth
            n_dims += 1
            n_match += match
            rows.append((d, str(predicted), str(truth), "corrected" if corrected else ""))
    w = max(len(r[0]) for r in rows)
    for d, p, t, c in rows:
        flag = "==" if p == t else "!="
        print(f"  {d:{w}s}  {p!s:30.30s} {flag} {t!s:30.30s}  {c}")
    print(f"\nfirst-prediction accuracy over asked dims: {n_match}/{n_dims} = {n_match/n_dims:.0%}")
    corrections = [u for u in user_inputs
                   if (u.get("payload") or {}).get("state") == "preference_correction_response"]
    print(f"correction-page responses this meal: {len(corrections)}")
    score("first-prediction accuracy", "GREEN", f"{n_match}/{n_dims} asked dims matched")
    score("corrections-to-converge", "PARTIAL",
          "per-meal counts computed; the convergence curve needs >=2 days of meals")
else:
    score("first-prediction accuracy", "RED", "no preference_predicted/asked events")

if repredicted:
    for e in repredicted:
        print(f"  triggers={e.get('triggers')} changed={list(e.get('changed', {}))} "
              f"kept_external={e.get('kept_external')}")
    n_changed = sum(len(e.get("changed", {})) for e in repredicted)
    score("correction propagation", "GREEN",
          f"{len(repredicted)} repredict passes moved {n_changed} dim values")
else:
    score("correction propagation", "RED",
          "no preference_repredicted events (needs meal recorded >= 48d3308)")

  robot_speed                                         medium                         == medium                          
  confirm_navigation_arrival                          yes (with auto-continue countd == yes (with auto-continue countd  
  confirm_manipulation                                yes (with auto-continue countd == yes (with auto-continue countd  
  wait_before_autocontinue_seconds                    30 sec                         == 30 sec                          
  microwave_time                                      1 min                          == 1 min                           
  skewering_axis                                      perpendicular to major axis    == perpendicular to major axis     
  confirm_feeding_pickup                              yes (with auto-continue countd == yes (with auto-continue countd  
  bite_dipping_preference                             do not dip                     == do not dip                      
  bite_ordering                 

## 2. User-side co-adaptation: engagement + response latency

Tap vs autocontinue on every side-channel-capable page (engaged agreement vs passive
timeout — the chat's blocker #1), and webapp response latency from the meal bag
(both `/robot_to_webapp` and `/webapp_to_robot` share one host clock).

In [4]:
acts = collections.Counter()
per_state = collections.defaultdict(collections.Counter)
for u in user_inputs:
    p = u.get("payload") or {}
    ua = p.get("user_action")
    if ua is not None:
        acts[ua] += 1
        per_state[p.get("state", "?")][ua] += 1

if acts:
    total = sum(acts.values())
    print(f"overall: {dict(acts)}  (tap share {acts.get('tap', 0)/total:.0%})")
    for st, c in sorted(per_state.items()):
        print(f"  {st:35s} {dict(c)}")
    score("tap-vs-autocontinue rates", "GREEN",
          f"{total} stamped responses, tap share {acts.get('tap', 0)/total:.0%}")
else:
    score("tap-vs-autocontinue rates", "RED", "no user_action stamps in user_inputs")

overall: {'tap': 37}  (tap share 100%)
  bite_confirm_transfer               {'tap': 1}
  bite_selection                      {'tap': 1}
  detection_confirm                   {'tap': 9}
  nav_adjust                          {'tap': 3}
  plate_release_confirm               {'tap': 3}
  preference_correction_response      {'tap': 20}
[GREEN  ] tap-vs-autocontinue rates — 37 stamped responses, tap share 100%


In [5]:
# Response latency: time from the last page-bearing /robot_to_webapp message to
# each /webapp_to_robot response. Page-bearing = parses as JSON with a 'state'
# key (the 1 Hz continuous explanations don't carry one), so this is a proxy —
# refine per page type by joining payload.state.
try:
    import rosbag
    bags = sorted(MEAL_BAG_DIR.glob("core_*.bag"))
    if not bags:
        raise FileNotFoundError(f"no core_*.bag in {MEAL_BAG_DIR}")
    pages, responses = [], []
    for b in bags:
        with rosbag.Bag(str(b)) as bag:
            for topic, msg, t in bag.read_messages(
                    topics=["/robot_to_webapp", "/webapp_to_robot"]):
                try:
                    d = json.loads(msg.data)
                except Exception:
                    d = {}
                if topic == "/robot_to_webapp":
                    if isinstance(d, dict) and "state" in d:
                        pages.append(t.to_sec())
                else:
                    responses.append(t.to_sec())
    pages.sort()
    lat = []
    import bisect
    for r in responses:
        i = bisect.bisect_left(pages, r)
        if i:
            lat.append(r - pages[i - 1])
    if lat:
        lat.sort()
        q = lambda f: lat[int(f * (len(lat) - 1))]
        print(f"n={len(lat)} responses | median {q(.5):.1f}s | p25 {q(.25):.1f}s | "
              f"p75 {q(.75):.1f}s | max {q(1.):.1f}s")
        score("response latency", "PARTIAL",
              f"median {q(.5):.1f}s over {len(lat)} responses (page-bearing-message proxy)")
    else:
        score("response latency", "RED", "no joinable page->response pairs in bag")
except ImportError:
    score("response latency", "SKIP", "rosbag not importable in this kernel")
except FileNotFoundError as e:
    score("response latency", "RED", str(e))

n=32 responses | median 0.1s | p25 0.1s | p75 0.3s | max 19.8s
[PARTIAL] response latency — median 0.1s over 32 responses (page-bearing-message proxy)


## 3. Skill success table (`skill_execute`, one event per attempt)

Attempts / success / takeover / failed per HLA, with verbatim failure reasons.
Falls back to `bt_timeline.csv` durations (no outcomes) for pre-`ef964a7` meals.

In [6]:
skills = by_cat.get("skill_execute", [])
if skills:
    tab = collections.defaultdict(collections.Counter)
    durs = collections.defaultdict(list)
    reasons = []
    for e in skills:
        tab[e["hla"]][e["outcome"]] += 1
        durs[e["hla"]].append(e.get("duration_s", 0.0))
        if e.get("failure_reason"):
            reasons.append((e["hla"], e["failure_reason"]))
    w = max(len(h) for h in tab)
    print(f"  {'HLA':{w}s}  attempts success takeover failed aborted  mean_s")
    for h, c in sorted(tab.items()):
        n = sum(c.values())
        print(f"  {h:{w}s}  {n:8d} {c['success']:7d} {c['takeover']:8d} "
              f"{c['failed']:6d} {c['aborted']:7d}  {statistics.mean(durs[h]):6.1f}")
    for h, r in reasons:
        print(f"  FAILURE {h}: {r}")
    n_all = sum(sum(c.values()) for c in tab.values())
    n_ok = sum(c["success"] for c in tab.values())
    score("skill success table", "GREEN", f"{n_ok}/{n_all} attempts succeeded")
else:
    score("skill success table", "RED",
          "no skill_execute events (needs meal recorded >= ef964a7)")
    if NAVLOG:
        rows = read_csv(NAVLOG / "bt_timeline.csv")
        starts = {}
        durs = []
        for r in rows:
            line = r.get("line", "")
            if line.startswith("Starting node: "):
                starts[line.split(": ", 1)[1]] = float(r["t_wall"])
            elif line.startswith("Finished executing node: "):
                node = line.split(": ", 1)[1]
                if node in starts:
                    durs.append((node, float(r["t_wall"]) - starts.pop(node)))
        for n, d in durs:
            print(f"  {n:28s} {d:7.1f}s")
        score("per-skill durations (fallback)", "GREEN",
              f"{len(durs)} node durations from bt_timeline.csv")

[RED    ] skill success table — no skill_execute events (needs meal recorded >= ef964a7)
  OpenFridge                      83.0s
  PickPlateFromAppliance          51.0s
  PlacePlateOnHolder              22.0s
  CloseFridge                     70.0s
  NavigateToMicrowave             46.0s
  OpenMicrowave                   83.0s
  PickPlateFromHolder             27.0s
  PlacePlateInAppliance           22.0s
  CloseMicrowave                  66.0s
  PressMicrowaveButton           109.0s
  OpenMicrowave                   90.0s
  PickPlateFromAppliance          59.0s
  PlacePlateOnHolder              21.0s
  CloseMicrowave                  64.0s
  NavigateToTable                139.0s
  GazeAtTable                     15.0s
  PickPlateFromHolder             27.0s
  PlacePlateOnTable               29.0s
  PickUtensil                     30.0s
  AcquireBite                     48.0s
  TransferUtensil                 63.0s
  StowUtensil                     24.0s
  PickPlateFromTable           

## 4. Bite-level agency (`bite_event`, one per acquisition attempt)

Autonomous vs manual mode, prediction-override rate (user disagreed with FLAIR),
dip usage, per-stage success.

In [7]:
bites = by_cat.get("bite_event", [])
if bites:
    modes = collections.Counter(b.get("mode") for b in bites)
    overrides = [b for b in bites
                 if b.get("mode") == "autonomous" and b.get("predicted_item")
                 and b.get("chosen_item") != b.get("predicted_item")]
    dips = [b for b in bites if b.get("dip")]
    ok = sum(bool(b.get("success")) for b in bites)
    print(f"attempts: {len(bites)} | modes: {dict(modes)} | success: {ok}/{len(bites)}")
    print(f"prediction overrides: {len(overrides)} | dipped bites: {len(dips)}")
    for b in bites:
        print(f"  {b.get('mode', '?'):17s} chosen={b.get('chosen_item')} "
              f"predicted={b.get('predicted_item')} dip={b.get('dip')} "
              f"success={b.get('success')}" + (f" ERROR={b['error']}" if b.get("error") else ""))
    score("bite-level agency", "GREEN",
          f"{len(bites)} attempts, {len(overrides)} overrides, {len(dips)} dips")
else:
    score("bite-level agency", "RED",
          "no bite_event events (needs meal recorded >= 48d3308)")

[RED    ] bite-level agency — no bite_event events (needs meal recorded >= 48d3308)


## 5. Interventions (teleop takeovers + micro-adjustments)

Takeovers with interrupted-HLA context come from `skill_execute` outcome rows;
nav micro-adjustments from the `nav_adjust` webapp inputs; the user-level
`teleop_intervention_log.jsonl` adds provenance where present.

In [8]:
takeovers = [e for e in by_cat.get("skill_execute", []) if e.get("outcome") == "takeover"]
nav_adj = [u for u in user_inputs if (u.get("payload") or {}).get("state") == "nav_adjust"]
tele_log = jsonl(USER_DIR / "teleop_intervention_log.jsonl")

if takeovers:
    for e in takeovers:
        print(f"  takeover during {e['hla']} (attempt {e.get('attempt')}) -> "
              f"{'redo' if e.get('redo') else 'continue-past'}")
    score("teleop takeovers by HLA", "GREEN", f"{len(takeovers)} takeovers with context")
else:
    score("teleop takeovers by HLA", "RED",
          "no skill_execute takeover rows (needs meal recorded >= ef964a7)")

print(f"nav micro-adjustments: {len(nav_adj)}")
print(f"teleop_intervention_log entries (user-level, cumulative): {len(tele_log)}")
score("nav micro-adjustments", "GREEN" if nav_adj else "PARTIAL",
      f"{len(nav_adj)} nav_adjust inputs")

[RED    ] teleop takeovers by HLA — no skill_execute takeover rows (needs meal recorded >= ef964a7)
nav micro-adjustments: 6
teleop_intervention_log entries (user-level, cumulative): 0
[GREEN  ] nav micro-adjustments — 6 nav_adjust inputs


## 6. Meal timeline & phase durations

Wall-clock skill timeline from the navlogger's `bt_timeline.csv`; meal window from
`metadata.json`; task commands from events.

In [9]:
if metadata.get("started") and metadata.get("ended"):
    span = metadata["ended"]["epoch"] - metadata["started"]["epoch"]
    print(f"meal window: {metadata['started']['iso']} -> {metadata['ended']['iso']} "
          f"({span/60:.0f} min)")
for e in by_cat.get("task_command", []):
    print(f"  task_command @ {e.get('iso', '?')}: "
          f"{ {k: v for k, v in e.items() if k not in ('epoch', 'iso', 'category', 'context')} }")
if NAVLOG and (NAVLOG / "bt_timeline.csv").exists():
    n = max(0, len(read_csv(NAVLOG / "bt_timeline.csv")))
    score("meal timeline / phase durations", "GREEN",
          f"{n} bt_timeline rows + metadata window")
else:
    score("meal timeline / phase durations", "PARTIAL", "no navlog bt_timeline found")

  task_command @ 2026-07-15T22:29:32.648499: {'task': 'meal_assistance', 'task_type': 'bite'}
  task_command @ 2026-07-15T22:49:16.305105: {'task': 'finish_feeding', 'task_type': 'place_plate_in_sink'}
[GREEN  ] meal timeline / phase durations — 54 bt_timeline rows + metadata window


## 7. Environment drift & calibration churn

`state_snapshot` events (every calibration-pickle rewrite, timestamped) plus the
observation events (`preference_color_recorded`, `preference_nav_offset_recorded`)
that watch the world shift under the robot.

In [10]:
snaps = by_cat.get("state_snapshot", [])
if snaps:
    per_file = collections.Counter(e.get("name") for e in snaps)
    for name, n in sorted(per_file.items()):
        print(f"  {name}: {n} snapshots")
    score("calibration churn", "GREEN", f"{sum(per_file.values())} snapshots")
else:
    score("calibration churn", "RED",
          "no state_snapshot events (needs meal recorded >= 6ed0f8f)")

for cat in ("preference_color_recorded", "preference_nav_offset_recorded"):
    for e in by_cat.get(cat, []):
        keys = {k: v for k, v in e.items() if k not in ("epoch", "iso", "category", "context")}
        print(f"  {cat} @ {e.get('iso', '?')}: {json.dumps(keys, default=str)[:120]}")
n_obs = len(by_cat.get("preference_color_recorded", [])) + \
        len(by_cat.get("preference_nav_offset_recorded", []))
score("environment observations", "GREEN" if n_obs else "PARTIAL",
      f"{n_obs} color/nav-offset observations")

[RED    ] calibration churn — no state_snapshot events (needs meal recorded >= 6ed0f8f)
  preference_color_recorded @ 2026-07-15T22:14:32.129685: {"location": "fridge", "field": "plate_color_fridge", "color": "h=12,s=223,v=169,range=0.10", "changed": false}
  preference_color_recorded @ 2026-07-15T22:24:27.936048: {"location": "microwave", "field": "plate_color_microwave", "color": "h=12,s=223,v=169,range=0.10", "changed": false}
  preference_color_recorded @ 2026-07-15T22:50:20.220814: {"location": "table", "field": "plate_color_table", "color": "h=12,s=223,v=169,range=0.10", "changed": false}
  preference_nav_offset_recorded @ 2026-07-15T22:16:52.010949: {"location": "microwave", "field": "nav_offset_microwave", "offset": "dx=0.000,dy=0.000,dyaw=0.000", "changed": false}
  preference_nav_offset_recorded @ 2026-07-15T22:28:11.655097: {"location": "table", "field": "nav_offset_table", "offset": "dx=0.000,dy=0.000,dyaw=0.000", "changed": false}
  preference_nav_offset_recorded @ 2026-07

## 8. Navigation performance

Per-goal duration and terminal residuals from the navlogger's `goals.csv`
(`settled` rows re-measured against a goal after the *next* one started are
excluded — known logger artifact).

In [11]:
if NAVLOG and (NAVLOG / "goals.csv").exists():
    rows = read_csv(NAVLOG / "goals.csv")
    done = [r for r in rows if r["event"] == "succeeded" and r.get("duration_s")]
    for r in done:
        print(f"  goal ({float(r['goal_x']):6.2f},{float(r['goal_y']):6.2f}) "
              f"{float(r['duration_s']):6.1f}s residual {float(r['residual_xy_m'])*100:4.1f}cm "
              f"/ {math.degrees(float(r['residual_yaw_rad'])):4.1f}deg")
    res = [float(r["residual_xy_m"]) for r in done]
    if res:
        print(f"\n{len(done)} goals | mean residual {statistics.mean(res)*100:.1f}cm")
    score("navigation performance", "GREEN", f"{len(done)} goals with residuals")
else:
    score("navigation performance", "PARTIAL", "no goals.csv in navlog")

  goal (  1.43, -0.12)   21.8s residual  4.2cm /  0.1deg
  goal (  1.43, -0.12)    0.6s residual  4.4cm /  0.4deg
  goal ( -0.12,  0.08)   31.5s residual  2.7cm /  1.2deg
  goal (  0.72, -4.34)   85.2s residual  3.0cm /  1.6deg
  goal (  0.72, -4.34)    0.2s residual  3.9cm /  1.6deg
  goal ( -0.35, -0.13)   80.8s residual  4.4cm /  0.5deg
  goal (  1.43, -0.09)   31.7s residual  4.0cm /  0.7deg
  goal (  1.43, -0.09)    0.2s residual  3.4cm /  0.4deg

8 goals | mean residual 3.7cm
[GREEN  ] navigation performance — 8 goals with residuals


## 9. Forced vs chosen participation

Forced = the interaction the system demanded (ask-flow correction pages).
Chosen = interaction the user initiated (settings edits, manual bite modes,
nav adjustments). The co-adaptation claim: forced trends to zero, chosen persists.

In [12]:
forced = len([u for u in user_inputs
              if (u.get("payload") or {}).get("state", "").startswith("preference_correction")])
settings_edits = len(by_cat.get("preference_settings_edit", []))
manual_bites = len([b for b in by_cat.get("bite_event", [])
                    if str(b.get("mode", "")).startswith("manual")])
chosen = settings_edits + manual_bites + len(nav_adj)
print(f"forced (ask-flow pages answered): {forced}")
print(f"chosen (settings {settings_edits} + manual bites {manual_bites} "
      f"+ nav adjust {len(nav_adj)}): {chosen}")
score("forced-vs-chosen split", "PARTIAL",
      f"forced={forced} chosen={chosen}; trend needs multiple days")

forced (ask-flow pages answered): 23
chosen (settings 0 + manual bites 0 + nav adjust 6): 6
[PARTIAL] forced-vs-chosen split — forced=23 chosen=6; trend needs multiple days


## 10. Scorecard

The dry-run verdict. RED rows name exactly what the next recorded meal must
demonstrate; a fully GREEN/PARTIAL board on a post-`48d3308` lab meal means the
dataset is sufficient **by construction**.

In [13]:
order = {"GREEN": 0, "PARTIAL": 1, "SKIP": 2, "RED": 3}
w = max(len(m) for m in SCORECARD)
for m, (s, note) in sorted(SCORECARD.items(), key=lambda kv: order.get(kv[1][0], 9)):
    print(f"{s:7s}  {m:{w}s}  {note}")
n = collections.Counter(s for s, _ in SCORECARD.values())
print(f"\n{n.get('GREEN', 0)} green / {n.get('PARTIAL', 0)} partial / "
      f"{n.get('SKIP', 0)} skip / {n.get('RED', 0)} red")
if n.get("RED"):
    print("RED cells above are expected against pre-2026-07-16 meals; "
          "re-run against the next lab meal to close them.")

GREEN    first-prediction accuracy        20/20 asked dims matched
GREEN    tap-vs-autocontinue rates        37 stamped responses, tap share 100%
GREEN    per-skill durations (fallback)   27 node durations from bt_timeline.csv
GREEN    nav micro-adjustments            6 nav_adjust inputs
GREEN    meal timeline / phase durations  54 bt_timeline rows + metadata window
GREEN    environment observations         6 color/nav-offset observations
GREEN    navigation performance           8 goals with residuals
PARTIAL  corrections-to-converge          per-meal counts computed; the convergence curve needs >=2 days of meals
PARTIAL  response latency                 median 0.1s over 32 responses (page-bearing-message proxy)
PARTIAL  forced-vs-chosen split           forced=23 chosen=6; trend needs multiple days
RED      correction propagation           no preference_repredicted events (needs meal recorded >= 48d3308)
RED      skill success table              no skill_execute events (needs meal rec